In [1]:
import requests

UNIPROT_CHRM3 = "P20309"  # human M3 (CHRM3)
CUTOFF_NM = 10000         # z.B. <= 10 µM

url = (
    "https://bindingdb.org/rest/getLigandsByUniprot"
    f"?uniprot={UNIPROT_CHRM3};{CUTOFF_NM}"
    "&response=application/json"
)

r = requests.get(url, timeout=60)
r.raise_for_status()
data = r.json()

# Struktur kann je nach Endpoint variieren -> erst inspizieren:
print(type(data))
print(list(data)[:10] if isinstance(data, dict) else data[:1])

<class 'dict'>
['getLindsByUniprotResponse']


In [2]:
resp = data["getLindsByUniprotResponse"]
print(type(resp))
print(resp.keys() if isinstance(resp, dict) else "not a dict")


<class 'dict'>
dict_keys(['bdb.hit', 'bdb.length', 'bdb.uniprot_length', 'bdb.primary', 'bdb.alternative', 'bdb.affinities'])


In [3]:
import numpy as np
import pandas as pd

resp = data["getLindsByUniprotResponse"]
raw = pd.json_normalize(resp["bdb.affinities"], sep=".")
print("[raw]", raw.shape)
print(raw.columns)

# --- rename auf einheitliche Namen ---
df = raw.rename(columns={
    "bdb.monomerid": "monomerid",
    "bdb.smile": "smiles",
    "bdb.affinity_type": "type",
    "bdb.affinity": "value",
}).copy()

df["type"] = df["type"].astype(str).str.upper().str.strip()
df["smiles"] = df["smiles"].astype(str).str.strip()
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# --- Annahme: nM (weil cutoff in nM definiert) ---
# pX = 9 - log10(nM)
df["value_nM"] = df["value"]
df["pX"] = 9 - np.log10(df["value_nM"])

# --- Optional: nur KI ---
df_ki = df[df["type"].eq("KI")].copy()
print("[KI only]", df_ki.shape)

# --- Dedup / Aggregation pro Verbindung (SMILES als Key) ---
# Hinweis: SMILES können in seltenen Fällen nicht-kanonisch sein -> besser wäre InChIKey via RDKit (unten)
agg = (
    df_ki
    .replace({"smiles": {"nan": np.nan, "None": np.nan, "": np.nan}})
    .dropna(subset=["smiles", "pX"])
    .groupby("smiles", as_index=False)
    .agg(
        n_measurements=("pX", "size"),
        median_pKi=("pX", "median"),
        min_pKi=("pX", "min"),
        max_pKi=("pX", "max"),
        monomerid=("monomerid", "first"),
    )
    .sort_values(["n_measurements", "median_pKi"], ascending=[False, False])
)

print("[agg]", agg.shape)
print(agg.head(10))

[raw] (2749, 4)
Index(['bdb.monomerid', 'bdb.smile', 'bdb.affinity_type', 'bdb.affinity'], dtype='object')
[KI only] (1763, 6)
[agg] (1598, 6)
                                                 smiles  n_measurements  \
113   CC(C)[N+]1([C@H]2CC[C@@H]1CC(C2)OC(=O)[C@@H](C...               3   
651   CN1[C@H]2CC[C@@H]1C[C@@H](C2)OC(=O)C(CO)c1cccc...               2   
247               CCN(CC)CCOC(=O)C(C)(c1ccccc1)c1ccccc1               2   
41    C1CN2CCC1C(C2)=C1c2ccccc2-c2ccccc12 |(7.74,-3....               2   
1457      OCCOCCN1CCN(CC1)C1=Nc2ccccc2Sc2ccccc12 |t:13|               2   
844   COc1ccccc1C(O)C1=CN2CCC1CC2 |t:11,(14.36,.8,;1...               2   
843   COc1ccccc1C(=O)C1=CN2CCC1CC2 |t:11,(11.44,1.23...               2   
1383  O=Cc1ccc(s1)C1=CN2CCC1CC2 |t:8,(13.96,-3.07,;1...               2   
1443  OC1(C2=CN3CCC2CC3)c2ccccc2CCc2ccccc12 |t:2,(8....               2   
1030  C[N+]1(C)[C@@H]2C[C@H](C[C@@H]1[C@@H]1O[C@H]21...               1   

      median_pKi    min_pKi    

In [4]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi

def clean_smiles(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if "|" in s:
        s = s.split("|", 1)[0].strip()
    return s or None

def smiles_to_inchikey(s):
    s = clean_smiles(s)
    if not s:
        return None
    try:
        m = Chem.MolFromSmiles(s)
        if m is None:
            return None
        return inchi.MolToInchiKey(m)
    except Exception:
        return None

# df_ki: dein KI-only DF aus dem REST Pull (mit Spalten: smiles, pX, monomerid, ...)
df_ki = df_ki.copy()
df_ki["smiles_clean"] = df_ki["smiles"].apply(clean_smiles)
df_ki["inchikey"] = df_ki["smiles_clean"].apply(smiles_to_inchikey)

print("InChIKey coverage:", df_ki["inchikey"].notna().mean())

bdb_agg = (
    df_ki.dropna(subset=["inchikey", "pX"])
        .groupby("inchikey", as_index=False)
        .agg(
            n_measurements=("pX", "size"),
            median_pKi=("pX", "median"),
            min_pKi=("pX", "min"),
            max_pKi=("pX", "max"),
            smiles=("smiles_clean", "first"),
            monomerid=("monomerid", "first"),
        )
)

print("[bdb_agg]", bdb_agg.shape)
bdb_agg.head(5)

InChIKey coverage: 0.9994327850255247
[bdb_agg] (1593, 7)


,inchikey,n_measurements,median_pKi,min_pKi,max_pKi,smiles,monomerid
0,AANNUKSOCVVCRY-UHFFFAOYSA-N,1,7.008774,7.008774,7.008774,NCCCCCN1CCC(CC1)OC(=O)Nc1ccccc1-c1ccccc1,50337868
1,AAPXNHMQKBDDJN-PGRDOPGGSA-N,1,5.689944,5.689944,5.689944,Cc1ncoc1-c1nnc(SCCCN2CC[C@@]3(C[C@H]3c3ccccc3)...,50192034
2,AASAEQLTIUBPQN-QLKFWGTOSA-N,1,9.698970,9.698970,9.698970,CN(C(=O)CCN1CCC(CC1)OC(=O)Nc1ccccc1-c1ccccc1)c...,103757
3,AAWDKUPZMHITDM-BHVANESWSA-N,1,9.100000,9.100000,9.100000,CN(CCN1CCC(CC1)OC(=O)Nc1ccccc1-c1ccccc1)C(=O)c...,50084427
4,AAZZRQOLTDMXGV-UHFFFAOYSA-N,1,5.309804,5.309804,5.309804,CN(CC#CCN1CCCC1)C(=O)CCCCCCCNC(=O)OC(C)(C)C,50368049


In [7]:
import pandas as pd
import numpy as np

CHEMBL_PATH = r"C:\Users\Besitzer\Desktop\M3_Project\M3_databases\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"
chembl = pd.read_csv(CHEMBL_PATH)

# --- finde SMILES-Spalte robust ---
if "smiles" in chembl.columns:
    chembl_smiles_col = "smiles"
elif "canonical_smiles" in chembl.columns:
    chembl_smiles_col = "canonical_smiles"
elif "SMILES" in chembl.columns:
    chembl_smiles_col = "SMILES"
else:
    chembl_smiles_col = None

# --- InChIKey vorhanden? sonst berechnen (nutzt deine smiles_to_inchikey + clean_smiles) ---
if "inchikey" not in chembl.columns:
    if chembl_smiles_col is None:
        raise RuntimeError("Keine SMILES-Spalte in ChEMBL gefunden (smiles/canonical_smiles/SMILES).")
    chembl["inchikey"] = chembl[chembl_smiles_col].apply(smiles_to_inchikey)

print("ChEMBL shape:", chembl.shape)
print("ChEMBL InChIKey coverage:", chembl["inchikey"].notna().mean())

ChEMBL shape: (2268, 14)
ChEMBL InChIKey coverage: 1.0


In [9]:
import numpy as np
import pandas as pd

# ---------------------------
# 0) Inputs (bereits vorhanden)
#   - chembl: (2268, 14), inchikey coverage 1.0
#   - bdb_agg: (1593, 7)  columns: inchikey, smiles, monomerid, median_pKi, min_pKi, max_pKi, n_measurements
# ---------------------------

OUT_DIR = r"C:\Users\Besitzer\Desktop\M3_Project\M3_databases"

# ---------------------------
# 1) Prepare BindingDB table for merge
# ---------------------------
bdb = bdb_agg.copy()

bdb = bdb.rename(columns={
    "smiles": "bdb_smiles",
    "monomerid": "bdb_monomerid",
    "median_pKi": "bdb_median_pKi",
    "min_pKi": "bdb_min_pKi",
    "max_pKi": "bdb_max_pKi",
    "n_measurements": "bdb_n",
})

bdb["bdb_source"] = "BindingDB"
bdb["bdb_target_uniprot"] = "P20309"

# ---------------------------
# 2) Merge (Annotation)
# ---------------------------
merged = chembl.merge(bdb, on="inchikey", how="left")
print("Merged shape:", merged.shape)
print("BindingDB coverage on ChEMBL:", float(merged["bdb_median_pKi"].notna().mean()))

OUT_ANN = OUT_DIR + r"\ChEMBL_M3_plus_BindingDB_annotated.csv"
merged.to_csv(OUT_ANN, index=False)
print("[SAVED]", OUT_ANN)

# ---------------------------
# 3) Extension: add new BindingDB ligands not present in ChEMBL
# ---------------------------
chembl_keys = set(chembl["inchikey"].unique())
new_bdb = bdb[~bdb["inchikey"].isin(chembl_keys)].copy()
print("New BindingDB ligands (not in ChEMBL):", new_bdb.shape[0])

def label_from_pki(pki, hi=6.0, lo=5.0):
    if pd.isna(pki):
        return np.nan
    if pki >= hi:
        return "active_bdb"
    if pki <= lo:
        return "inactive_bdb"
    return "ambiguous_bdb"

new_bdb["consensus_label"] = new_bdb["bdb_median_pKi"].apply(label_from_pki)

# pick a smiles column name that exists in ChEMBL for the appended rows
if "smiles" in chembl.columns:
    smiles_col = "smiles"
elif "canonical_smiles" in chembl.columns:
    smiles_col = "canonical_smiles"
elif "SMILES" in chembl.columns:
    smiles_col = "SMILES"
else:
    smiles_col = "smiles"  # fallback; will be created

new_rows = pd.DataFrame({
    "inchikey": new_bdb["inchikey"],
    smiles_col: new_bdb["bdb_smiles"],
    "consensus_label": new_bdb["consensus_label"],
    "source_db": "BindingDB",
    "target_uniprot": "P20309",
    "bdb_median_pKi": new_bdb["bdb_median_pKi"],
    "bdb_min_pKi": new_bdb["bdb_min_pKi"],
    "bdb_max_pKi": new_bdb["bdb_max_pKi"],
    "bdb_n": new_bdb["bdb_n"],
    "bdb_monomerid": new_bdb["bdb_monomerid"],
})

# Extended = annotated ChEMBL rows + appended BindingDB-only rows
extended = pd.concat([merged, new_rows], ignore_index=True, sort=False)
print("Extended shape:", extended.shape)

OUT_EXT = OUT_DIR + r"\ChEMBL_M3_extended_with_BindingDB.csv"
extended.to_csv(OUT_EXT, index=False)
print("[SAVED]", OUT_EXT)

OUT_NEW_ONLY = OUT_DIR + r"\BindingDB_new_ligands_only.csv"
new_rows.to_csv(OUT_NEW_ONLY, index=False)
print("[SAVED]", OUT_NEW_ONLY)

# ---------------------------
# 4) Conflicts report (ChEMBL label vs BindingDB pKi)
# ---------------------------
def chembl_coarse(lbl):
    if pd.isna(lbl):
        return np.nan
    s = str(lbl).lower()
    # your labels include active_single/inactive_single/active/inactive
    if "inactive" in s:
        return "inactive"
    if "active" in s:
        return "active"
    if "ambig" in s:
        return "ambiguous"
    return np.nan

merged2 = merged.copy()
merged2["chembl_coarse"] = merged2["consensus_label"].apply(chembl_coarse)

def bdb_coarse(pki, hi=6.0, lo=5.0):
    if pd.isna(pki):
        return np.nan
    if pki >= hi:
        return "active"
    if pki <= lo:
        return "inactive"
    return "ambiguous"

merged2["bdb_coarse"] = merged2["bdb_median_pKi"].apply(bdb_coarse)

conflicts = merged2[
    merged2["bdb_coarse"].notna() &
    merged2["chembl_coarse"].notna() &
    (merged2["bdb_coarse"] != "ambiguous") &
    (merged2["chembl_coarse"] != "ambiguous") &
    (merged2["bdb_coarse"] != merged2["chembl_coarse"])
].copy()

print("Conflicts:", conflicts.shape[0])

OUT_CONFLICTS = OUT_DIR + r"\ChEMBL_vs_BindingDB_conflicts.csv"
conflicts.to_csv(OUT_CONFLICTS, index=False)
print("[SAVED]", OUT_CONFLICTS)

# ---------------------------
# 5) Quick sanity stats
# ---------------------------
if new_bdb.shape[0] > 0:
    print("New BindingDB median pKi summary:")
    print(new_bdb["bdb_median_pKi"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

Merged shape: (2268, 22)
BindingDB coverage on ChEMBL: 0.4025573192239859
[SAVED] C:\Users\Besitzer\Desktop\M3_Project\M3_databases\ChEMBL_M3_plus_BindingDB_annotated.csv
New BindingDB ligands (not in ChEMBL): 680
Extended shape: (2948, 24)
[SAVED] C:\Users\Besitzer\Desktop\M3_Project\M3_databases\ChEMBL_M3_extended_with_BindingDB.csv
[SAVED] C:\Users\Besitzer\Desktop\M3_Project\M3_databases\BindingDB_new_ligands_only.csv
Conflicts: 0
[SAVED] C:\Users\Besitzer\Desktop\M3_Project\M3_databases\ChEMBL_vs_BindingDB_conflicts.csv
New BindingDB median pKi summary:
count    680.000000
mean       6.661678
std        1.218494
min        5.010000
1%         5.058280
5%         5.209533
50%        6.259637
95%        9.190992
99%       10.164372
max       11.008774
Name: bdb_median_pKi, dtype: float64
